# Comparing Changes in Tycho's Supernova Remnant Across Time
<hr style="border: 2px solid #f5bf03" />

- **Description:** Tutorial on comparing changes in Tycho's Supernova Remnant over many years.
- **Level:** Advanced
- **Data:** XMM observations of SN 1572 (obsid=Multiple)
- **Requirements:** Must be run using pySAS version 2.3.0 or higher.
- **Credit:** Ryan Tanner (March 2026)
- **Support:** <a href="https://heasarc.gsfc.nasa.gov/docs/xmm/xmm_helpdesk.html">XMM Newton GOF Helpdesk</a>
- **Last verified to run:** 15 March 2026, for SAS v22.1 and pySAS v2.3.0

<hr style="border: 2px solid #f5bf03" />

## 1. Introduction

This tutorial uses data from multiple observations. The data will be stored in the user's `data_dir` as automatically configured by pySAS. The data for each Obs ID will be placed in individual subdirectories inside the `data_dir`. Each Obs ID will have its own "`work`" directory where all files created by running SAS tasks will be stored. pySAS has been designed to allow rapid shifting between different Obs IDs. All environment variables needed by SAS will automatically be set by pySAS when a new Obs ID is made "active". pySAS will automatically detect which files are present and store paths and filenames for all files associated with an Obs ID.

The Obs IDs we will be working with will be stored in a list named `obsids`. To work with each Obs ID we use a `for` loop on the `obsids` list. For each Obs ID we will create an "`instance`" ("`my_pps`") of the `PPSFiles` object, as shown below.

```python
for obsid in obsids:
    my_pps = pysas.PPSFiles(obsid)
```

Any SAS task we run will operate *only* on the currently active Obs ID. Each time an Obs ID is activated pySAS will change the "current working directory" to the "`work`" directory of that Obs ID.

In this tutorial we will only be using data from the MOS cameras, not the pn. This is because the pn data has significant out-of-time contamination, which, while we can filter that out, would add significant complexity to an already complex tutorial. It is left as an exercise to the interested reader to do the analysis using pn data.

#### Useful Links

- [`pysas` Documentation](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/pysas/index.html "pysas Documentation")
- [`pysas` on GitHub](https://github.com/XMMGOF/pysas)
- [Common SAS Threads](https://www.cosmos.esa.int/web/xmm-newton/sas-threads "SAS Threads")
- [Users' Guide to the XMM-Newton Science Analysis System (SAS)](https://xmm-tools.cosmos.esa.int/external/xmm_user_support/documentation/sas_usg/USG/SASUSG.html "Users' Guide")
- [The XMM-Newton ABC Guide](https://heasarc.gsfc.nasa.gov/docs/xmm/abc/ "ABC Guide")
- [XMM Newton GOF Helpdesk](https://heasarc.gsfc.nasa.gov/docs/xmm/xmm_helpdesk.html "Helpdesk") - Link to form to contact the GOF Helpdesk.

<div class="alert alert-block alert-warning">
    <b>Warning:</b> By default this notebook will place observation data files in your default <tt>data_dir</tt> directory. Make sure pySAS has been configured properly.
</div>

In [ ]:
# pySAS imports
import pysas
from pysas import MyTask
pysas.sas_cfg.set_setting('pysas_verbosity','WARNING')

# HEASoftpy import
import heasoftpy as hsp

# Generic VO access routines
import pyvo as vo

# Useful imports
import os, re, shutil

# Imports for plotting
import astropy
import matplotlib.pyplot as plt
from astropy.visualization import astropy_mpl_style
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.wcs import WCS
from astropy.table import Table
plt.style.use(astropy_mpl_style)

# To handle certain warnings
import warnings
warnings.filterwarnings('ignore')

Filenames to be used in this notebook.

In [ ]:
instruments = ['EMOS1','EMOS2']

filtered_evtls      = {}
gti_file            = {}
time_filtered_evtls = {}
merged_event_lists  = {}
time_filt_image     = {}
obs_year            = {}

for inst in instruments:
    filtered_evtls[inst]      = f'{inst}_filtered_event_list.fits'
    gti_file[inst]            = f'{inst}_gti_rate.fits'
    time_filtered_evtls[inst] = f'{inst}_time_filtered_event_list.fits'
    merged_event_lists[inst]  = f'{inst}_merged_event_list.fits'
    time_filt_image[inst]     = f'{inst}_time_filt_image.fits'

In [ ]:
def make_hires_image(in_event_list,
                     zcolumn  = None,
                     out_image='image.fits',
                     output   = True):

    inargs = {'table'         : in_event_list, 
              'withimageset'  : 'yes',
              'imageset'      : out_image,
              'imagebinning'  : 'binSize',
              'xcolumn'       : 'X',
              'ycolumn'       : 'Y',
              'ximagebinsize' : 40,
              'yimagebinsize' : 40}

    if zcolumn is not None:
        inargs['zcolumn'] = zcolumn
    
    MyTask('evselect', inargs, output_to_terminal = output).run()

def plot_light_curve(light_curve_file,rate_cut=None,title=None,log=False):
    ts = Table.read(light_curve_file,hdu=1)
    fig, ax = plt.subplots()
    ax.plot(ts['TIME'],ts['RATE'])
    if rate_cut:
         ax.axhline(y=rate_cut, color='b')
    if log:
        ax.set_yscale('log')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Count Rate (ct/s)')
    if title is not None:
        ax.set_title(title)
    fig.show()

## 2. Download PPS Files

This next cell will do a `TAP` search for XMM observations within 0.1 degrees of SN 1572. It will also only return observations where the MOS was in Full Frame mode with the medium filter. This will return a list of 11 Obs IDs that we can work with. We will then sort the Obs IDs based on when they were taken to have the list in time order.

The tutorial [Using PyVO to Find Observations for Analysis](./misc-xmm-using-pyvo-to-find-obsids.ipynb) explains in depth how this query was put together.

In [ ]:
target = SkyCoord.from_name('SN 1572')
heasarc_tap = vo.regsearch(servicetype='tap',keywords=['heasarc'])[0]
query = """SELECT obsid, time
           FROM xmmmaster as cat 
           WHERE cat.mos1_mode LIKE 'FF-ME%' and 
                 cat.mos2_mode LIKE 'FF-ME%' and 
                 cat.pn_mode LIKE 'FF-ME%' and 
                 contains(point('ICRS',cat.ra,cat.dec),circle('ICRS',{},{},0.1))=1
        """.format(target.ra.deg, target.dec.deg)
obsid_table = heasarc_tap.search(query).to_table()
obsid_table.sort('time')

obsids = []
for row in obsid_table:
    obsids.append(row['obsid'])

obsid_table

If you uncomment the lines in the next cell, you can erase all of the data for the Obs IDs in the list above. This will "reset" the associated Obs ID directories in your data directory.

In [ ]:
# for obsid in obsids:
#     my_pps = pysas.PPSFiles(obsid, output_to_terminal=False)
#     my_pps.clear_obs_dir()

We will be using the `Pipeline Processing System` (`PPS`) files for this tutorial. Each Obs ID can have anywhere between about a dozen PPS files to a few thousand PPS files. Rather than download **ALL** of the potentially thousands of PPS files, we will only download the four files that we need.

This next cell does two things:

1. Collects the PPS filenames of files we will need.
2. Downloads **only** the PPS files we need.

To get the filenames we need, we first use the function `get_list_of_all_filenames` which returns a list of all PPS filenames for a single Obs ID *without downloading any files*. 

Then we use the function `return_filenames_from_product_dict` which returns the filenames of specific PPS product types based on the list of PPS filenames we procured previously.

Once these filenames are collected into a single list they are passed to the `download_PPS_data` function to download just the PPS files we need (Note: The input parameter `filename` can take a single filename, or a list of filenames).

In [ ]:
for obsid in obsids:
    print(f'Downloading event lists for Obs ID: {obsid}')
    my_pps = pysas.PPSFiles(obsid)
    list_pps_files  = my_pps.get_list_of_all_filenames()
    mos_event_lists = my_pps.return_filenames_from_product_dict(my_pps.EPIC_products['MIEVLI_FIT'], list_of_files=list_pps_files)
    attitude_file   = my_pps.return_filenames_from_product_dict(my_pps.Obs_products['ATTTSR_FIT'], list_of_files=list_pps_files)
    light_curves    = my_pps.return_file_list_on_pattern('.*(M1|M2).*FBKTSR.*FTZ', list_of_files=list_pps_files)
    list_of_files   = mos_event_lists + attitude_file + light_curves
    my_pps.download_PPS_data(filename=list_of_files)

This next cell will run `cifbuild` so that we have the current calibrations for our data.

In [ ]:
for obsid in obsids:
    my_pps = pysas.PPSFiles(obsid)
    my_pps.run_cifbuild()

In [ ]:
for obsid in obsids:
    my_pps = pysas.PPSFiles(obsid, output_to_terminal=False)
    print(f'Obs ID: {obsid}')
    for event_list in my_pps.EPIC_event_lists:
        with fits.open(event_list) as hdu:
            inst     = hdu[0].header['INSTRUME']
            exposure = hdu[0].header['EXPIDSTR']
            ontime   = hdu[1].header['ONTIME']
            obs_date = hdu[0].header['DATE-OBS']
        obs_year[obsid] = obs_date[0:4]
        if 'S' in exposure:
            sch = 'Scheduled'
        else:
            sch = 'Unscheduled'
        print(f'    {inst} {sch:>11} Exposure : Live Time = {ontime:.0f} s')

In [ ]:
obs_year

The first observation (0096210101) experienced an inturruption and thus the scheduled exposure only has 1700-1800 seconds worth of data. Then the observation was completed with an unscheduled exposure with 20,000 seconds worth of data. Thus we will use the unscheduled exposure for our analysis. We will build a list of the event lists we will use, along with the names of other files we will use.

We will store the filenames in two dictionaries, `event_lists` for the event lists, and `ltcv_files` for the light curves.

In [ ]:
event_lists = {}
ltcv_files  = {}

# This block of code handles ObsID=0096210101 since we only want the 
# "unscheduled" exposure.
obsid = obsids[0]
event_lists[obsid] = {}
ltcv_files[obsid]  = {}
my_pps = pysas.PPSFiles(obsid, output_to_terminal=False)
for event_list in my_pps.EPIC_event_lists:
    with fits.open(event_list) as hdu:
        inst = hdu[0].header['INSTRUME']
        exposure = hdu[0].header['EXPIDSTR']
    if 'U' in exposure: event_lists[obsid][inst] = event_list
    light_curves = my_pps.return_file_list_on_pattern('.*(M1|M2).*FBKTSR.*FTZ')
    for light_curve in light_curves:
        with fits.open(light_curve) as hdu:
            inst     = hdu[0].header['INSTRUME']
            exposure = hdu[0].header['EXPIDSTR']
        if 'U' in exposure: ltcv_files[obsid][inst] = light_curve

# This block of code handles the rest of the Obs IDs.
for obsid in obsids[1:]:
    event_lists[obsid] = {}
    ltcv_files[obsid]  = {}
    my_pps = pysas.PPSFiles(obsid, output_to_terminal=False)
    for event_list in my_pps.EPIC_event_lists:
        with fits.open(event_list) as hdu:
            inst = hdu[0].header['INSTRUME']
        event_lists[obsid][inst] = event_list
    light_curves = my_pps.return_file_list_on_pattern('.*(M1|M2).*FBKTSR.*FTZ')
    for light_curve in light_curves:
        with fits.open(light_curve) as hdu:
            inst     = hdu[0].header['INSTRUME']
            exposure = hdu[0].header['EXPIDSTR']
        ltcv_files[obsid][inst] = light_curve

In [ ]:
for obsid in obsids:
    print(f'Obs ID: {obsid}')
    for inst in instruments:
        print(f'    {inst} event list : {os.path.basename(event_lists[obsid][inst])}')
        print(f'    {inst} light curve: {os.path.basename(ltcv_files[obsid][inst])}')

## 3. Quick Look at the Data

We will now do a quick look at the data by first looking at basic images created from the event lists, and then looking at the light curves.

In [ ]:
low_res_image_files = {}

for obsid in obsids:
    low_res_image_files[obsid] = {}
    my_pps = pysas.PPSFiles(obsid, output_to_terminal=False)
    for inst, event_list in event_lists[obsid].items():
        low_res_image_files[obsid][inst] = f'{inst}_image.fits'
        my_pps.quick_eplot(event_list, title=f'{inst} Image for Obs ID: {obsid}', image_file=low_res_image_files[obsid][inst], vmax=1000.0)

The light curves that come with the PPS files have been partially processed. The brightest sources have been removed and the curves show the in-field-of-view events, covering the 0.5-7.5 keV band.

In [ ]:
for obsid in obsids:
    for inst in instruments:
        plot_light_curve(ltcv_files[obsid][inst],title=f'{inst} Light Curve for Obs ID: {obsid}')

By scanning over the light curves we see that most of them have some level of contamination from solar flares. We will have to filter the data and see how much of each Obs ID contains useful data. We will filter the light curves based on `RATE` to generate a GTI file to apply to each event list. We chose the rate based on inspecting each light curve and choosing an appropriate rate. Because of variablility the rate has to be set for each Obs ID and for each MOS camera.

<div class="alert alert-block alert-info">
    <b>Note:</b> We are making some general assumptions about what is a "good" rate for filtering for each instrument for each Obs ID. Different assumptions can be made and the interested user is encouraged to explore them.
</div>

In [ ]:
rate_cuts = {'0096210101' : {'EMOS1' : 17, 'EMOS2' : 13},
             '0310590101' : {'EMOS1' : 10, 'EMOS2' : 9 },
             '0310590201' : {'EMOS1' : 10, 'EMOS2' : 9 },
             '0412380101' : {'EMOS1' : 8 , 'EMOS2' : 8 },
             '0412380201' : {'EMOS1' : 8 , 'EMOS2' : 8 },
             '0511180101' : {'EMOS1' : 10, 'EMOS2' : 10},
             '0412380301' : {'EMOS1' : 7 , 'EMOS2' : 8 },
             '0801840201' : {'EMOS1' : 6 , 'EMOS2' : 7 },
             '0801840301' : {'EMOS1' : 6 , 'EMOS2' : 7 },
             '0801840401' : {'EMOS1' : 5 , 'EMOS2' : 8 },
             '0801840501' : {'EMOS1' : 6 , 'EMOS2' : 8 }}

We will replot the light curves with the rate cut limits included on the plots.

In [ ]:
for obsid in obsids:
    for inst in instruments:
        plot_light_curve(ltcv_files[obsid][inst],rate_cut=rate_cuts[obsid][inst],title=f'{inst} Light Curve for Obs ID: {obsid}')

## 4. Filtering the Data

We start by applying a basic filter to the event lists. This will fix some of the contamination. But then we will have to create good time interval (GTI) files to filter out times of high solar activity. This will leave us with an estimate of how much useful data each Obs ID contains.

In [ ]:
def filter_event_list(in_event_list,
                      out_event_list,
                      pi_min,
                      pi_max):

    with fits.open(in_event_list) as hdu:
        inst = hdu[0].header['INSTRUME']

    if inst == 'EPN':
        filter = 'XMMEA_EP'
        pattern = 4
    elif 'EMOS' in inst:
        filter = 'XMMEA_EM'
        pattern = 12

    # Filter expression
    expression = '(PATTERN in [0:{pattern}])&&(PI in [{pi_min}:{pi_max}])&&(FLAG == 0)&&#{filter}'.format(filter=filter,pattern=pattern,pi_min=pi_min,pi_max=pi_max)

    inargs = {'table'           : in_event_list, 
              'withfilteredset' : 'yes', 
              "expression"      : expression, 
              'filteredset'     : out_event_list, 
              'filtertype'      : 'expression', 
              'keepfilteroutput': 'yes', 
              'updateexposure'  : 'yes', 
              'filterexposure'  : 'yes'}
    
    MyTask('evselect', inargs).run()

In [ ]:
for obsid in obsids:
    my_pps = pysas.PPSFiles(obsid, output_to_terminal=False)
    for inst, event_list in event_lists[obsid].items():
        filter_event_list(event_list,filtered_evtls[inst],300,10000)

In [ ]:
def apply_gti(light_curve_file,in_event_list,gti_rate_file,rate_cut,out_event_list):
    with fits.open(light_curve_file) as hdu:
        inst     = hdu[0].header['INSTRUME']
        obsid    = hdu[0].header['OBS_ID']

    inargs = {'table'      : light_curve_file, 
              'gtiset'     : gti_rate_file,
              'timecolumn' : 'TIME', 
              "expression" : "'(RATE <= {0})'".format(rate_cut)}
    
    MyTask('tabgtigen', inargs, output_to_terminal=False).run()

    useful_data = False

    with fits.open(gti_rate_file) as hdu:
        ontime = hdu[1].header['ONTIME']
    print(f'{obsid}-{inst:<5} : On Time = {ontime}')
    if ontime > 500:
        useful_data = True
        
        inargs = {'table'           : in_event_list,
                  'withfilteredset' : 'yes', 
                  "expression"      : "'GTI({0},TIME)'".format(gti_rate_file), 
                  'filteredset'     : out_event_list,
                  'filtertype'      : 'expression', 
                  'keepfilteroutput': 'yes',
                  'updateexposure'  : 'yes', 
                  'filterexposure'  : 'yes'}
        
        MyTask('evselect', inargs, output_to_terminal=False).run()

    return useful_data

We will now filter the event lists based on `Rate`. This will remove contamnation from solar flares. Some Obs IDs are entirely contaminated and will have no useable data after filtering. We will make a new list of Obs IDs with useful data using the cutoff that there must be at least 500 seconds of usable data. The following cell will display the total amount of usable time (`On Time`) after filtering.

In [ ]:
good_event_lists = {}

for obsid in obsids:
    good_event_lists[obsid] = {}
    my_pps = pysas.PPSFiles(obsid, output_to_terminal=False)
    for inst in instruments:
        useful_data = apply_gti(ltcv_files[obsid][inst],filtered_evtls[inst],gti_file[inst],rate_cuts[obsid][inst],time_filtered_evtls[inst])
        if useful_data:
            good_event_lists[obsid][inst] = time_filtered_evtls[inst]
        else:
            good_event_lists[obsid][inst] = None
    my_pps.resolve_obs_dir()

In [ ]:
for obsid in obsids:
    print(f'Obs ID: {obsid}')
    for inst in instruments:
        print(f'    {inst} good event list : {good_event_lists[obsid][inst]}')

## 5. Generating Images

Now that the event lists are filtered for solar flares let's take a look at the fitlered event lists.

In [ ]:
for obsid in obsids:
    my_pps = pysas.PPSFiles(obsid, output_to_terminal=False)
    for inst,event_list in good_event_lists[obsid].items():
        if event_list is None: continue
        my_pps.quick_eplot(event_list, title=f'{inst} Image for Obs ID: {obsid}', image_file=time_filt_image[inst], vmax=1000.0)

We use the task `etruecolor` to create "flat" images across three energy bands; 0.3-0.7 keV, 0.7-1.2 keV, and 1.2-7.0 keV.

In [ ]:
def run_etruecolor(event_lists,att_file,out_image):
    
    inargs = {'tablelist' : event_lists,
              'attfile'   : att_file, 
              'min'       : 300,
              'max'       : 7000, 
              'fileset'   : out_image}
    
    MyTask('etruecolor', inargs).run()

In [ ]:
for obsid in obsids:
    my_pps = pysas.PPSFiles(obsid, output_to_terminal=False)
    list_of_files = []
    for inst,event_list in good_event_lists[obsid].items():
        if event_list is None: continue
        list_of_files.append(event_list+':EVENTS')
    input_string = ' '.join(list_of_files)
    run_etruecolor(' '.join(list_of_files), my_pps.attitude_file, 'color_image.fits')

Now we collect some filenames for the images we will be plotting.

In [ ]:
image_files = []

for obsid in obsids:
    my_pps = pysas.PPSFiles(obsid, output_to_terminal=False)
    tempdict = {'filename'    : os.path.join(my_pps.work_dir,'color_image.fits'), 
                'obsid'       : obsid, 
                'final_image' : os.path.join(my_pps.work_dir,f'{obsid}_final_plot.png')}
    image_files.append(tempdict)

The function in the next cell will plot all three energy bands side-by-side and will center the images on a common coordinate.

In [ ]:
def plot_images(image_file,titles, vmin=0.00001, vmax=0.05, RA=0.0, DEC=0.0, out_plot=None):
    """
    Takes a tricolor image and plots the three bands in a row.
    """
    # Set image limits
    ra_ll  = RA_OBJ-0.55
    ra_ul  = RA_OBJ+0.3
    dec_ll = DEC_OBJ-0.15
    dec_ul = DEC_OBJ+0.15
    lims = [[ra_ll, dec_ll, 0],
            [ra_ul, dec_ul, 0]]
    
    hdu = fits.open(image_file)
    image_hdu = hdu[0]
    wcs = WCS(image_hdu.header)
    coords = wcs.all_world2pix(lims, 0)

    # fig, axes = plt.subplots(1, 3,subplot_kw={'projection': WCS(hdu[0].header), 'slices':('x', 'y', 0)}, figsize=(15, 15))
    fig, axes = plt.subplots(1, 3,subplot_kw={'projection': wcs, 'slices':('x', 'y', 0)}, figsize=(15, 5))
    plt.grid(color='blue', ls='solid')
    
    for i, ax in enumerate(axes.flat):
        ax.set_facecolor("black")
        # ax.grid(color='blue', ls='solid')
        ax.grid(False)
        ax.imshow(image_hdu.data[i, :, :], origin='lower', norm='log', vmin=vmin, vmax=vmax)
        ax.set(xlim=(coords[1][0], coords[0][0]), ylim=(coords[0][1], coords[1][1]))
        ax.title.set_text(titles[i])
        ax.coords[0].set_axislabel('RA')
        if i != 0:
            ax.coords[1].set_ticklabel_visible(False)
            ax.coords[1].set_axislabel('')
        else:
            ax.coords[1].set_axislabel('Dec')
    
    plt.subplots_adjust(wspace=0)

    # Save the plot
    if out_plot:
        plt.savefig(out_plot)

In [ ]:
with fits.open(good_event_lists[obsids[0]]['EMOS1']) as hdu:
    RA_OBJ  = hdu[0].header['RA_OBJ']
    DEC_OBJ = hdu[0].header['DEC_OBJ']
    
for image_file in image_files:
    titles = ['0.3-0.7 keV','0.7-1.2 keV','1.2-7.0 keV']
    plot_images(image_file['filename'], titles, RA=RA_OBJ, DEC=DEC_OBJ, out_plot=image_file['final_image'])

To compare the changes we have taken the first and final images and created an animated gif to show the expansion of the shock wave.

![Change in SN Remnant](./_files/Tycho_SN_Remnant_XMM_Beg_End.gif)